In [ ]:
import MDAnalysis as mda
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

In [ ]:
def load_secondary_structure_ranges(filename):
    """
    Load secondary structure ranges from a file.
    Args:
        filename: Path to the file containing the secondary structure ranges
    Returns:
        ss_residues: Set of residue indices that are in secondary structure elements, in the 1-based index
    """
    ss_residues = []
    with open(filename) as f:
        for line in f:
            if line.strip():
                parts = line.split()
                #start, end in 0-based index
                start = int(parts[1]) - 1
                end = int(parts[2]) - 1
                ss_residues.append(start)
                ss_residues.append(end)
    ss_residues = sorted(ss_residues)
    return ss_residues

def compute_chirality_gpu(traj, normalize=True):

    """
    Compute local chirality parameters across a protein chain using PyTorch
    for each frame in the trajectory.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    traj_tensor = torch.tensor(traj, dtype=torch.float32, device=device)

    v1 = traj_tensor[:, 1:-2] - traj_tensor[:, :-3]
    v2 = traj_tensor[:, 2:-1] - traj_tensor[:, 1:-2]
    v3 = traj_tensor[:, 3:]   - traj_tensor[:, 2:-1]

    cross = torch.cross(v2, v3, dim=2)
    dot = torch.sum(v1 * cross, dim=2)
    
    if normalize:
        epsilon = 1e-8
        norm_v1 = torch.norm(v1, dim=2)
        norm_v2 = torch.norm(v2, dim=2)
        norm_v3 = torch.norm(v3, dim=2)
        denom = norm_v1 * norm_v2 * norm_v3 + epsilon # avoid divide-by-zero
        chi = dot / denom
    else:
        chi = dot    

    return chi.cpu().numpy()

# def compute_chirality(u, selection='all'):
#     """
#     Compute local chirality parameters across a protein chain
#     for each frame in the trajectory.
#     """
#     chirality_all = []
#     for ts in u.trajectory:
#         positions = u.select_atoms(selection).positions
#         N = len(positions)
#         chi_vals = []
#         for i in range(N - 3):
#             v1 = positions[i+1] - positions[i]
#             v2 = positions[i+2] - positions[i+1]
#             v3 = positions[i+3] - positions[i+2]
#             chi = np.dot(v1, np.cross(v2, v3))
#             chi_vals.append(chi)
#         chirality_all.append(chi_vals)
#     return np.array(chirality_all)  # shape: (n_frames, n_residues - 3)

def compute_chirality(traj, normalize=True):
    """
    Compute local chirality across trajectory using NumPy vectorization.
    
    Parameters:
        traj: np.ndarray of shape (n_frames, n_residues, 3)
        normalize: bool, if True, normalize the chirality values
        epsilon: float, small value to avoid division by zero in normalization

    Returns:
        chi: np.ndarray of shape (n_frames, n_residues - 3)
    """
    # Bond vectors
    v1 = traj[:, 1:-2, :] - traj[:, :-3, :]
    v2 = traj[:, 2:-1, :] - traj[:, 1:-2, :]
    v3 = traj[:, 3:, :]   - traj[:, 2:-1, :]

    # Compute cross product and scalar triple product
    cross = np.cross(v2, v3, axis=2)
    dot = np.einsum('ijk,ijk->ij', v1, cross)

    if normalize:
        epsilon=1e-8
        norm_v1 = np.linalg.norm(v1, axis=2)
        norm_v2 = np.linalg.norm(v2, axis=2)
        norm_v3 = np.linalg.norm(v3, axis=2)
        denom = norm_v1 * norm_v2 * norm_v3 + epsilon  # prevent divide-by-zero
        chi = dot / denom
    else:
        chi = dot

    return chi

In [ ]:
# validation function
def compute_signed_agreement(chi_sim, chi_ref):
    """
    Computes the signed agreement between trajectory chirality and reference.
    Parameters:
        chi_sim: np.ndarray, shape (n_frames, n_residues-3)
        chi_ref: np.ndarray, shape (n_residues-3,)
    Returns:
        np.ndarray, shape (n_frames,)
    """
    sign_sim = np.sign(chi_sim)
    sign_ref = np.sign(chi_ref)
    # return np.mean(sign_sim * sign_ref, axis=1)
    
    # Compare signs (True where they match, False otherwise)
    match = sign_sim == sign_ref
    
    # Compute fraction of matches per frame
    return np.mean(match, axis=1)

def compute_chirality_rmsd(chi_sim, chi_ref):
    """
    Computes RMSD between simulation chirality and reference.
    Parameters:
        chi_sim: np.ndarray, shape (n_frames, n_residues-3)
        chi_ref: np.ndarray, shape (n_residues-3,)
    Returns:
        np.ndarray, shape (n_frames,)
    """
    return np.sqrt(np.mean((chi_sim - chi_ref[None, :])**2, axis=1))#/len(chi_ref)



def export_match_profile(per_residue_match, output_csv="chirality_match_profile.csv"):
    """
    Saves per-residue match profile to CSV.
    """
    df_match = pd.DataFrame({
        "Residue Index": np.arange(len(per_residue_match)) + 2,
        "Match Fraction": per_residue_match
    })
    df_match.to_csv(output_csv, index=False)
    print(f"Saved match profile to {output_csv}")

def detect_mirror_segments(chi_sim, chi_ref):
    """
    Detect mirror-like behavior per residue by comparing simulation average
    and sign consistency to reference chirality.
    Returns:
        delta_chi: difference of average chi from reference (N-3,)
        sign_match_fraction: how often sign matches reference (N-3,)
    """
    avg_chi = np.mean(chi_sim, axis=0)
    delta_chi = np.sign(avg_chi) - np.sign(chi_ref)

    sign_sim = np.sign(chi_sim)
    sign_ref = np.sign(chi_ref)
    sign_match_fraction = np.mean(sign_sim == sign_ref[None, :], axis=0)
    
    return delta_chi, sign_match_fraction

# def plot_chirality_analysis(signed_agreement, chirality_rmsd, delta_chi, per_residue_match):
#     """
#     Visualizes chirality metrics using matplotlib.
#     """
#     fig, axs = plt.subplots(4, 1, figsize=(12, 10), constrained_layout=True)

#     axs[0].plot(signed_agreement)
#     axs[0].set_title("Signed Agreement S(t) with Reference")
#     axs[0].set_xlabel("Frame")
#     axs[0].set_ylabel("Mean Sign Match")
#     axs[0].set_ylim([-1, 1])
#     axs[0].axhline(0, color='gray', linestyle='--')

#     axs[1].plot(chirality_rmsd)
#     axs[1].set_title("Chirality RMSD Time Series")
#     axs[1].set_xlabel("Frame")
#     axs[1].set_ylabel("RMSD")

#     # Plot per-residue mirror signature
#     # axs[2].figure(figsize=(12, 4))
#     axs[2].bar(np.arange(len(delta_chi)), delta_chi)
#     axs[2].axhline(0, color='gray', linestyle='--')
#     axs[2].set_title("Mean Chirality Deviation from Reference per Residue")
#     axs[2].set_xlabel("Residue Index (Offset by +2)")
#     axs[2].set_ylabel("Δχᵢ = sign(⟨χᵢ⟩) - sign(χᵢ(ref))")
    
    
#     axs[3].bar(np.arange(len(per_residue_match)), per_residue_match)
#     axs[3].set_title("Per-Residue Chirality Match with Reference")
#     axs[3].set_xlabel("Residue Index (Offset by +2)")
#     axs[3].set_ylabel("Fraction of Frames with Matching Sign")
#     axs[3].set_ylim([0, 1])
#     axs[3].axhline(0.5, color='gray', linestyle='--')



#     plt.tight_layout()
#     plt.show()

def plot_chirality_analysis(signed_agreement, chirality_rmsd, delta_chi, per_residue_match):
    """
    Visualizes chirality metrics in a 2x2 subplot layout using matplotlib.
    """
    fig, axs = plt.subplots(2, 2, figsize=(14, 6), constrained_layout=True)

    # Plot signed agreement
    axs[0, 0].plot(signed_agreement)
    axs[0, 0].set_title("Signed Agreement S(t) with Ref. (higher=better)")
    axs[0, 0].set_xlabel("Frame")
    axs[0, 0].set_ylabel("Mean Sign Match")
    axs[0, 0].set_ylim([0, 1])
    # axs[0, 0].axhline(0, color='gray', linestyle='--')

    # Plot chirality RMSD
    axs[1, 0].plot(chirality_rmsd)
    axs[1, 0].set_title("Chirality RMSD Time Series  (lower=better)")
    axs[1, 0].set_xlabel("Frame")
    axs[1, 0].set_ylabel("RMSD")
    axs[1, 0].set_ylim([0, 1])

    # Mean deviation per residue
    axs[0, 1].bar(np.arange(len(delta_chi)), delta_chi)
    axs[0, 1].axhline(0, color='gray', linestyle='--')
    axs[0, 1].set_title("Mean Chirality Deviation per Residue (lower=better)")
    axs[0, 1].set_xlabel("Residue Index (Offset by +2)")
    axs[0, 1].set_ylabel("Δχᵢ = sign(⟨χᵢ⟩) - sign(χᵢ(ref))")

    # Match fraction per residue
    axs[1, 1].bar(np.arange(len(per_residue_match)), per_residue_match)
    axs[1, 1].set_title("Per-Residue Chirality Match with Ref. (higher=better)")
    axs[1, 1].set_xlabel("Residue Index (Offset by +2)")
    axs[1, 1].set_ylabel("Match Fraction")
    axs[1, 1].set_ylim([0, 1])
    axs[1, 1].axhline(0.5, color='gray', linestyle='--')

    plt.show()

In [ ]:
def process_chirality(psf, ref_cor, traj_dcd, sec_def=None):
    selection = 'all'
    ref_u = mda.Universe(psf, ref_cor, format='CRD')
    u = mda.Universe(psf, traj_dcd)
    if sec_def: # calculate for secondary element only
        atm_indices = load_secondary_structure_ranges(sec_def)
        #reference
        ref_pos = np.array([ref_u.select_atoms(selection).positions[atm_indices]])
        # trajectory
        ag = u.select_atoms(selection)
        n_atoms = len(atm_indices)
        n_frames = len(u.trajectory)
        traj = np.empty((n_frames, n_atoms, 3), dtype=np.float32)
        for i, ts in enumerate(u.trajectory):
            traj[i] = ag.positions[atm_indices]
    else: #all atoms in topopoly

        #convert to shape of 1xn_residues x 3 for consistent with the input requirement of compute_chirality_ function
        ref_pos = np.array([ref_u.select_atoms(selection).positions]) 
        # combine trajectories coordinate to numpy array for torch
        ag = u.select_atoms(selection)
        n_atoms = len(ag)
        n_frames = len(u.trajectory)
        traj = np.empty((n_frames, n_atoms, 3), dtype=np.float32)
        for i, ts in enumerate(u.trajectory):
            traj[i] = ag.positions

    chiral_ref = compute_chirality(ref_pos)[0] #ref is single frame, get the first array for consistence with the requirement of validation function
    chiral_traj = compute_chirality(traj)
    # === Run analysis ===
    signed_agreement = compute_signed_agreement(chiral_traj, chiral_ref)
    chirality_rmsd = compute_chirality_rmsd(chiral_traj, chiral_ref)
    delta_chi, sign_match_fraction = detect_mirror_segments(chiral_traj, chiral_ref)

    return signed_agreement, chirality_rmsd, delta_chi, sign_match_fraction
    
    # plot_chirality_analysis(signed_agreement, chirality_rmsd, delta_chi, sign_match_fraction)

## Run analysis

In [ ]:
idx=1
signed_agreement, chirality_rmsd, delta_chi, sign_match_fraction = process_chirality('../template/setup/P38085_clean_ca.psf', '../template/setup/P38085_clean_ca.cor', f'../{idx}/P38085_{idx}_prod.dcd', '../template/setup/secondary_struc_defs.txt')

In [ ]:
plot_chirality_analysis(signed_agreement, chirality_rmsd, delta_chi, sign_match_fraction)